In [1]:
from fz import fz

fz_instance = fz()

# 1) Détection des variables
vars_found = fz_instance.ParseInput("SR-U-UN-G1-C4-N_lead.mcnp")
print("Variables détectées :", vars_found)

# 2) Compilation (substitution, exécution R, formatage)
fz_instance.CompileInput(
    input_file="SR-U-UN-G1-C4-N_lead.mcnp",
    input_variables={
        "rfis_cm": [8.65180], # fissile radius (cm)
        "wall_thk_cm": [1.0, 5.0, 10.0, 20.0], #[1.0, 5.0, 10.0, 20.0], # wall thickness (cm)
        "D_m": [1.0, 2.0, 5.0, 10.0, 20.0, 50.0, 100.0, 200.0, 300.0, 500.0, 700.0, 1000.0, 1200.0], #[1.0, 2.0, 5.0, 10.0, 20.0, 50.0, 100.0, 200.0, 300.0, 500.0, 700.0, 1000.0, 1200.0], # distance (m) from source edge to detector center
        # "L": [], # distance from source edge to wall edge (=D/2)
    },
    filename_template="{prefix}_{wall_thk_cm:.0f}cm_D{D_m:.0f}m.i", #"{prefix}_{wall_thk_cm:.0f}cm_D{D_m:.0f}m{ext}",
    use_dirs=True,
    group_variables=["rfis_cm"],  # Variables to group by
)

Variables détectées : {'rfis_cm', 'D_m', 'wall_thk_cm'}
Generated : wall_thk_cm=1.0/D_m=1.0/SR-U-UN-G1-C4-N_lead_1cm_D1m.i with {'wall_thk_cm': 1.0, 'D_m': 1.0, 'rfis_cm': 8.6518}
Generated : wall_thk_cm=1.0/D_m=2.0/SR-U-UN-G1-C4-N_lead_1cm_D2m.i with {'wall_thk_cm': 1.0, 'D_m': 2.0, 'rfis_cm': 8.6518}
Generated : wall_thk_cm=1.0/D_m=5.0/SR-U-UN-G1-C4-N_lead_1cm_D5m.i with {'wall_thk_cm': 1.0, 'D_m': 5.0, 'rfis_cm': 8.6518}
Generated : wall_thk_cm=1.0/D_m=10.0/SR-U-UN-G1-C4-N_lead_1cm_D10m.i with {'wall_thk_cm': 1.0, 'D_m': 10.0, 'rfis_cm': 8.6518}
Generated : wall_thk_cm=1.0/D_m=20.0/SR-U-UN-G1-C4-N_lead_1cm_D20m.i with {'wall_thk_cm': 1.0, 'D_m': 20.0, 'rfis_cm': 8.6518}
Generated : wall_thk_cm=1.0/D_m=50.0/SR-U-UN-G1-C4-N_lead_1cm_D50m.i with {'wall_thk_cm': 1.0, 'D_m': 50.0, 'rfis_cm': 8.6518}
Generated : wall_thk_cm=1.0/D_m=100.0/SR-U-UN-G1-C4-N_lead_1cm_D100m.i with {'wall_thk_cm': 1.0, 'D_m': 100.0, 'rfis_cm': 8.6518}
Generated : wall_thk_cm=1.0/D_m=200.0/SR-U-UN-G1-C4-N_lead_1c

In [2]:
import os
import stat
import csv
from pathlib import Path

# 1) Le CSV en entrée, généré par fz
csv_file = Path("generated_files.csv")

# 2) Le SLURM launcher que vous avez créé
slurm_script = Path("advantG_63.sh").resolve()

# 3) Le fichier de sortie
output_list = Path("list_sbatch_tasks.txt")

with csv_file.open("r", encoding="utf-8") as fin, output_list.open("w", encoding="utf-8") as fout:
    reader = csv.DictReader(fin)
    for row in reader:
        path = Path(row["path"].replace("\\", "/"))
        filename = row["file"]
        fout.write(f"(cd {path} && sbatch {slurm_script} {filename})\n")

# --- rendre le fichier exécutable pour l'utilisateur ---
# méthode 1 : forcer 0o755
output_list.chmod(0o755)

# méthode 2 : ne rajouter que le bit u+x au mode existant
# current_mode = output_list.stat().st_mode
# output_list.chmod(current_mode | stat.S_IXUSR)

print(f"✅ Fichier généré et chmod +x appliqué : {output_list}")


✅ Fichier généré et chmod +x appliqué : list_sbatch_tasks.txt


In [ ]:
# import csv
# from pathlib import Path

# # Entrée
# csv_file = Path("generated_files.csv")

# # Chemin absolu vers le script d’exécution
# script_path = Path("run_advantg_local.sh").resolve()

# # Fichier de sortie
# output_file = Path("list_glost_tasks.txt")

# with csv_file.open("r", encoding="utf-8") as f_in, output_file.open("w", encoding="utf-8") as f_out:
#     reader = csv.DictReader(f_in)
#     for row in reader:
#         # Nettoyage du chemin : remplacement \ par /
#         path = Path(row["path"].replace("\\", "/"))
#         filename = row["file"]
#         cmd = f"(cd {path} && {script_path} {filename})\n"
#         f_out.write(cmd)

# print(f"✅ Fichier {output_file} généré avec succès.")
